# Graph controls matched to a saved training run

**Role:** Training.

Benchmark a completed GIN or FiLM training run using its exact saved cohort, ordered train/validation folds, fate encoding, target transforms, global-feature scaling, and fitted baseline. Inherit the trained depth/width/seed grid and optimization settings. The original reference checkpoint bundles for the selected folds are copied byte-for-byte into the comparison; only control models are fitted. Select ring-based controls and/or retrain the reference architecture with permuted or constant fate inputs. Store prepared control inputs, models, reference provenance, overall MSE and regional depth-wise MSE in a new self-contained training run.


In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/models/gnn.py").is_file())  # Locate the repository from the notebook’s working directory.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ROOT = PROJECT_ROOT  # Repository root used to resolve data and result paths.


## Reference run and control choices

Select a completed run from `gin_depth_training` and one reference family (`gin` or `film`). Every saved depth, width and seed for that family is included for the selected folds. `max_folds=1` selects the lowest original fold ID; `None` uses all folds. Increasing the limit includes more original folds without generating a new holdout or changing any membership. Configure only which controls to train and the new run tag; model, dataset and optimization parameters are inherited below. `reference` means the selected reference architecture. Its intact trained checkpoint is always included without retraining, even if omitted from the requested controls.


In [ ]:
import copy
import hashlib
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from src.models.ring_mlp import RingFractionMLP, RingFractionSizeMLP
from src.data.ring_features import attach_precomputed_ring_features
from src.models.size_models import seed_all
from src.training.loop import TrainConfig, train
from src.training.losses import WeightedLossTerm, edge_loss_term
from src.inference.predict import predict_targets
from src.artifacts.bundle import save_bundle, load_bundle
from src.artifacts.checkpoints import build_model, model_spec
from src.artifacts.provenance import workflow_fingerprint
from src.artifacts.paths import training_run_path, resolve_training_run
from src.artifacts.runs import AnalysisRun
from src.analysis.metrics.evaluation import compare_baseline_mse
from src.training.progress import TrainingProgress


In [ ]:
# Execution
RUN_TRAINING = True  # True trains controls and saves a new run; the reference is never retrained.
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'  # Execution device; fitted reference data stay identical.

# Reference and requested controls
CONTROL_SETTINGS = dict(
    reference_notebook='gin_depth_training',  # Training notebook folder under training_results/.
    reference_run='scMultiplexpaper_20260918_161107',  # Run directory name or explicit path; None requires exactly one available run.
    reference_family='gin',  # Reference family in that run: gin or film; keep every saved depth/width/seed.
    max_folds=1,  # Maximum original folds to include, in ascending fold-ID order; None uses all.

    families=['reference', 'ring', 'ring-size'],  # reference uses the saved GIN/FiLM architecture.
    signals=['intact', 'permuted', 'constant'],  # Fixed whole-vector permutation or all-one fate inputs.

    tag='controls',  # Purpose label for the new training_results/graph_controls_training/ run.
)

# Destinations
REFERENCE_RUN = resolve_training_run(ROOT, CONTROL_SETTINGS['reference_notebook'], CONTROL_SETTINGS['reference_run'])
RUN_DIR = training_run_path(ROOT, 'graph_controls_training', CONTROL_SETTINGS['tag'])

# Progress display
SHOW_PROGRESS = True  # Show one persistent panel for models, epochs and completed tasks.
PRINT_EPOCHS = False  # Also print each epoch; False keeps the training output compact.


## Restore reference settings and model grid

Require a completed run with explicit cohort and per-fold input bundles. Load its settings rather than expose a second set of training parameters. Check that the selected intact, full-marker family has the complete saved depth/width/seed/fold grid. Apply the fold limit after validating the reference grid. The effective settings record the selected fold IDs and count, with the original count retained as `reference_n_folds`. The effective settings and original reference settings are both saved in the new output; they can be inspected before enabling training. Historical runs lacking this complete handoff are rejected rather than approximated.


In [ ]:
reference = AnalysisRun(REFERENCE_RUN)
if not reference.modern or not (REFERENCE_RUN/'complete.json').is_file():
    raise ValueError('Select a completed bundle-based training run with models.json and complete.json.')
family = CONTROL_SETTINGS['reference_family']
if family not in ('gin', 'film'):
    raise ValueError('reference_family must be gin or film.')
for name, allowed in [('families', {'reference', 'ring', 'ring-size'}),
                      ('signals', {'intact', 'permuted', 'constant'})]:
    values = CONTROL_SETTINGS[name]
    if not values or len(set(values)) != len(values) or not set(values) <= allowed:
        raise ValueError(f'Choose distinct {name} from {sorted(allowed)}.')
reference_records = reference.records[
    (reference.records.name == family) & (reference.records.subset == 'all')
    & (reference.records.signal == 'intact')].copy()
if reference_records.empty or 'input_bundle' not in reference_records:
    raise ValueError('The reference must contain intact, full-marker models with saved per-fold input bundles.')
reference_settings = copy.deepcopy(reference.settings)
SETTINGS = copy.deepcopy(reference_settings)
SETTINGS.update(tag=CONTROL_SETTINGS['tag'], families=CONTROL_SETTINGS['families'],
                signals=CONTROL_SETTINGS['signals'], marker_subsets={'all': None},
                reference_run=str(REFERENCE_RUN.resolve()), reference_family=family,
                control_settings=copy.deepcopy(CONTROL_SETTINGS))
membership = json.loads((REFERENCE_RUN/'splits.json').read_text())
fold_membership = {int(s['fold']): s for s in membership}
expected_grid = {(fold, depth, width, seed) for fold in fold_membership
                 for depth in SETTINGS['depths'] for width in SETTINGS['hidden_dims'] for seed in SETTINGS['seeds']}
actual_grid = set(reference_records[['fold', 'depth', 'hidden_dim', 'seed']].itertuples(index=False, name=None))
if actual_grid != expected_grid or len(reference_records) != len(expected_grid):
    raise ValueError('Reference model grid is incomplete or duplicated; finish the reference training first.')
max_folds = CONTROL_SETTINGS['max_folds']
if max_folds is not None and (isinstance(max_folds, bool) or not isinstance(max_folds, int) or max_folds < 1):
    raise ValueError('max_folds must be a positive integer or None.')
selected_folds = sorted(fold_membership)[:max_folds]
membership = [fold_membership[fold] for fold in selected_folds]
fold_membership = {s['fold']: s for s in membership}
reference_records = reference_records[reference_records.fold.isin(selected_folds)].sort_values(
    ['fold', 'depth', 'hidden_dim', 'seed'])
SETTINGS.update(reference_n_folds=reference_settings['n_folds'], n_folds=len(selected_folds),
                selected_folds=selected_folds)
print(f'Using original folds {selected_folds} from {reference_settings["n_folds"]} reference folds; no resplitting.')
cohort_payload = load_bundle(REFERENCE_RUN/'cohort')
marker_names = cohort_payload['all_marker_names']
cohort = pd.read_csv(REFERENCE_RUN/'cohort.csv')
region_settings_path = REFERENCE_RUN/'regional_evaluation/settings.json'
REGION_SETTINGS = dict(json.loads(region_settings_path.read_text()) if region_settings_path.exists() else {})
REGION_SETTINGS.pop('dataset', None)  # Dataset path is derived from the inherited run settings.
print('Reference:', REFERENCE_RUN)
print('Output:', RUN_DIR)
display(reference_records[['key', 'name', 'depth', 'hidden_dim', 'fold', 'seed']])
display(cohort.groupby('timepoint').agg(organoids=('organoid_str', 'size'), median_N=('n_cells', 'median')))

requested_controls = {(f, s) for f in CONTROL_SETTINGS['families'] for s in CONTROL_SETTINGS['signals']}
new_model_count = len(reference_records) * len(requested_controls - {('reference', 'intact')})
print(f"Planned: {new_model_count} new control models + {len(reference_records)} reused reference checkpoints.")


## Validate exact saved folds and preprocessing

Load each fold's prepared graphs and fitted transforms directly. Verify that graph order agrees with `splits.json`, that the full fate panel is present, and that baseline predictions were saved. No dataset loading, filtering, splitting, target interpolation, scaling fit, or baseline training is performed here. The same saved targets, globals, transforms, and baseline will be reused for every control.


In [ ]:
fold_inputs = {}
for fold, members in fold_membership.items():
    paths = reference_records.loc[reference_records.fold == fold, 'input_bundle'].unique()
    if len(paths) != 1:
        raise ValueError(f'Fold {fold} has multiple reference input bundles; select a homogeneous reference run.')
    prepared = load_bundle(REFERENCE_RUN/paths[0])
    required = {'groups', 'transform', 'baseline', 'baseline_offsets', 'baseline_predictions',
                'baseline_validation_mse', 'marker_names'}
    if not required <= prepared.keys() or prepared['baseline'] is None:
        raise ValueError(f'Fold {fold} is missing its fitted baseline or preprocessing artifacts.')
    if prepared['marker_names'] != marker_names:
        raise ValueError('The reference must use the complete saved marker panel.')
    for role, saved_role in [('train', 'train'), ('val', 'validation')]:
        ids = [g.organoid_str for g in prepared['groups'][role]]
        if ids != members[saved_role] or not ids or len(set(ids)) != len(ids):
            raise ValueError(f'Fold {fold}: {role} graph order differs from saved membership.')
    if set(members['train']) & set(members['validation']):
        raise ValueError(f'Fold {fold} has overlapping training/validation membership.')
    fold_inputs[fold] = prepared
support = pd.DataFrame([dict(fold=fold, role=role, organoid_str=org)
    for fold, members in fold_membership.items() for role in ('train', 'validation') for org in members[role]])
support = support.merge(cohort[['organoid_str', 'timepoint']], on='organoid_str', validate='many_to_one')
display(support.groupby(['fold', 'role', 'timepoint']).size().rename('organoids').to_frame())


## Train controls and preserve the reference

Save the reference cohort with only the selected fold memberships in `splits.json` (the original full split list is retained separately as `reference_splits.json`), and save a provenance record linking every control to its matching reference checkpoint. Copy the original intact reference model bundle byte-for-byte; no training call is made for it. Choosing `reference` with `permuted` or `constant` trains a new altered-signal control of the same architecture, not the original reference. For those and the ring controls, train requested controls from fresh initialization using the inherited architecture/grid and optimization settings. The permutation seed is the saved split seed plus fold, shared across all controls at that fold. Permutations shuffle whole fate vectors within each organoid in both training and validation; constant inputs replace fates with ones. Graph topology, targets, global features and preprocessing remain exact copies. Ring architectures change the aggregation rule by design while sharing applicable widths, dropout, normalization, globals and loss settings.

Exact-hop marker fractions and ring sizes are precomputed once on CPU per fold/signal, after permutation or constant-signal replacement, through the maximum saved depth. Both ring controls slice these fixed features to their own depth; caches are saved with the prepared inputs. This removes repeated neighborhood searches from training and evaluation. Changing fate inputs requires rebuilding these derived features.

The persistent progress panel reports the active model/depth/fold/seed (and signal, marker subset or masking rate), trained models remaining, epoch, best stopping metric, patience used and elapsed time. Baselines and copied/reused checkpoints have separate counters. The completion table updates in place and is also written to `training_progress.csv` in the run folder. Standard models stop on validation MAE; masking models stop on intact inner-validation MSE in transformed units. Epoch bars show the maximum budget, not an estimated completion time. `PRINT_EPOCHS=True` restores detailed epoch output. Interrupted or failed tasks are labelled and are not counted as completed.


In [ ]:
if not RUN_TRAINING:
    raise RuntimeError('Review the reference and control choices, then set RUN_TRAINING=True.')
RUN_DIR.mkdir(parents=True, exist_ok=False)
(RUN_DIR/'settings.json').write_text(json.dumps(SETTINGS, indent=2))
(RUN_DIR/'reference_settings.json').write_text(json.dumps(reference_settings, indent=2))
(RUN_DIR/'splits.json').write_text(json.dumps(membership, indent=2))
shutil.copy2(REFERENCE_RUN/'splits.json', RUN_DIR/'reference_splits.json')
shutil.copy2(REFERENCE_RUN/'cohort.csv', RUN_DIR/'cohort.csv')
save_bundle(RUN_DIR/'cohort', {**cohort_payload, 'settings': SETTINGS}, splits=membership,
            provenance={'reference_run': str(REFERENCE_RUN.resolve())})
if (REFERENCE_RUN/'exclusivity_rules.json').is_file():
    shutil.copy2(REFERENCE_RUN/'exclusivity_rules.json', RUN_DIR/'exclusivity_rules.json')
provenance = workflow_fingerprint(ROOT/'experiments/training/graph_controls_training.ipynb')
reference_files = ['settings.json', 'splits.json', 'models.json', 'cohort/manifest.json']
reference_files += [str(Path(path)/'manifest.json') for path in
                   set(reference_records.input_bundle) | set(reference_records.bundle)]
reference_provenance = dict(reference_run=str(REFERENCE_RUN.resolve()),
    reference_family=family, selected_folds=selected_folds, selected_keys=reference_records.key.tolist(),
    file_sha256={path: hashlib.sha256((REFERENCE_RUN/path).read_bytes()).hexdigest() for path in reference_files})
(RUN_DIR/'reference.json').write_text(json.dumps(reference_provenance, indent=2))
(RUN_DIR/'provenance.json').write_text(json.dumps({'code_sha256': provenance, 'device': DEVICE}, indent=2))
records, scores, histories, baseline_rows = [], [], {}, []
with TrainingProgress(new_model_count, total_copies=len(reference_records), show=SHOW_PROGRESS,
                      print_epochs=PRINT_EPOCHS, log_path=RUN_DIR/'training_progress.csv') as progress:
    for fold, prepared in fold_inputs.items():
        members = fold_membership[fold]
        baseline_rows.extend(dict(fold=fold, **row) for row in prepared['baseline_validation_mse'])
        pd.DataFrame(baseline_rows).to_csv(RUN_DIR/'baseline_validation_mse.csv', index=False)
        # Include the original intact reference even if only other controls were requested.
        signals = list(dict.fromkeys(['intact', *CONTROL_SETTINGS['signals']]))
        for signal in signals:
            groups = copy.deepcopy(prepared['groups'])
            rng_signal = np.random.default_rng(SETTINGS['split_seed'] + fold)
            for group in groups.values():
                for g in group:
                    if signal == 'permuted':
                        g.x = g.x[torch.as_tensor(rng_signal.permutation(len(g.x)))].clone()
                    elif signal == 'constant':
                        g.x = torch.ones_like(g.x)
            if signal in CONTROL_SETTINGS['signals'] and {'ring', 'ring-size'} & set(CONTROL_SETTINGS['families']):
                # These inputs are fixed: compute on CPU once after the signal edit, not on every forward pass.
                max_hops = int(reference_records.depth.max())
                progress.stage(f'Preparing ring features: fold {fold}, signal {signal}, through hop {max_hops}')
                for role, group in groups.items():
                    attach_precomputed_ring_features(group, k_hops=max_hops, inplace=True)
                # Save caches with the exact edited inputs for training and subsequent evaluation.
            input_bundle = f'inputs/fold_{fold}_all_{signal}'
            prepared_inputs = {**prepared, 'groups': groups}
            save_bundle(RUN_DIR/input_bundle, prepared_inputs, splits=members, provenance={'code_sha256': provenance})
            control_families = CONTROL_SETTINGS['families'] if signal in CONTROL_SETTINGS['signals'] else []
            if signal == 'intact':
                control_families = list(dict.fromkeys(['reference', *control_families]))
            for source_record in reference_records[reference_records.fold == fold].to_dict('records'):
                source_model = load_bundle(REFERENCE_RUN/source_record['bundle'])
                spec = model_spec(source_model['model'])
                for control_family in control_families:
                    depth, width, seed = (int(source_record[c]) for c in ('depth', 'hidden_dim', 'seed'))
                    reused = control_family == 'reference' and signal == 'intact'
                    name = family if control_family == 'reference' else control_family
                    with progress.task(kind='copied' if reused else 'model', model=name, signal=signal, depth=depth, width=width, fold=fold, seed=seed):
                        if reused:
                            model = source_model['model']
                            metrics, history = source_model['metrics'], source_model['history']
                        else:
                            seed_all(seed)
                            if control_family == 'reference':
                                model = build_model(spec)
                            else:
                                cls = {'ring': RingFractionMLP, 'ring-size': RingFractionSizeMLP}[control_family]
                                kw = {k: spec['kwargs'][k] for k in ('n_markers', 'hidden_dim', 'dropout', 'norm', 'global_dim')}
                                model = cls(**kw, k_hops=depth)
                            cfg = TrainConfig(**{k: SETTINGS[k] for k in
                                ['lr', 'weight_decay', 'batch_size', 'max_epochs', 'patience', 'num_workers', 'loss_name']},
                                device=DEVICE, aux_losses=[WeightedLossTerm(name='edge', fn=edge_loss_term,
                                    weight=SETTINGS['edge_loss_weight'], params=SETTINGS['edge_loss_params'])])
                            seed_all(seed)  # Match the reference training-loop seed after model construction.
                            model, metrics, history = train(model, groups['train'], groups['val'], cfg)
                        key = f'{name}_all_{signal}_d{depth}_h{width}_f{fold}_s{seed}'
                        record = dict(key=key, name=name, subset='all', signal=signal, depth=depth, hidden_dim=width,
                            fold=fold, seed=seed, bundle=f'models/{key}', input_bundle=input_bundle,
                            reference_key=source_record['key'], reused_reference=reused)
                        progress.stage('Evaluating held-out predictions and saving checkpoint')
                        y, mu, _ = predict_targets(groups['val'], model, target_transform=prepared['transform'], device=DEVICE)
                        start = 0
                        for g in groups['val']:
                            end = start + len(g.x)
                            scores.append(dict(**record, organoid_str=g.organoid_str, n_cells=len(g.x),
                                mse=float(np.mean((np.asarray(y)[start:end]-np.asarray(mu)[start:end])**2))))
                            start = end
                        if reused:
                            # Copy the original weights and bundle byte-for-byte; never fit this model again.
                            shutil.copytree(REFERENCE_RUN/source_record['bundle'], RUN_DIR/record['bundle'])
                        else:
                            save_bundle(RUN_DIR/record['bundle'], dict(model=model.cpu(), record=record, history=history, metrics=metrics),
                                        splits=members, provenance={'code_sha256': provenance, 'reference_key': source_record['key']})
                        records.append(record)
                        if not reused:
                            histories[key] = history
                        pd.DataFrame(records).to_json(RUN_DIR/'models.json', orient='records', indent=2)
                        compare_baseline_mse(scores, baseline_rows).to_csv(RUN_DIR/'validation_mse.csv', index=False)
    (RUN_DIR/'complete.json').write_text(json.dumps({'models': len(records), 'reference_run': str(REFERENCE_RUN)}))
    print('Saved run:', RUN_DIR)


## Compare controls with the original reference

Compare physical-curvature validation MSE across depth/ring radius. The intact reference curves use the existing trained checkpoints, while the other curves use new controls on the exact same folds and targets. The baseline is the copied reference baseline, never a refitted model. Bands are mean ± one SEM across organoids after averaging repeated seeds/validation appearances within each organoid. Optimization histories are shown only for newly trained controls. Reopen the new run with the standard `AnalysisRun` interface; copied bundles make its model/data loading independent of the reference directory.


In [ ]:
from src.analysis.metrics.evaluation import organoid_mse_summary
baseline_df = pd.DataFrame(baseline_rows)
scores_df = compare_baseline_mse(scores, baseline_rows)
display(scores_df.groupby(['name','subset','signal','depth'])[['mse','baseline_mse','mse_minus_baseline']].mean())
display(organoid_mse_summary(scores_df, ['name','subset','signal','hidden_dim','depth']))
display(organoid_mse_summary(baseline_df, ['fold']))
fig, axes = plt.subplots(1, 2, figsize=(12,4))
for label, frame in scores_df.groupby(['name','subset','signal','hidden_dim']):
    curve = organoid_mse_summary(frame, 'depth')
    line, = axes[0].plot(curve.depth, curve['mean'], 'o-', label=str(label))
    axes[0].fill_between(curve.depth, curve['mean'] - curve['sem'], curve['mean'] + curve['sem'],
                         color=line.get_color(), alpha=.18)
for key, history in histories.items():
    axes[1].plot(history['val_mae'], alpha=.5, label=key)
axes[0].set(xlabel='Depth / ring radius', ylabel='Mean organoid validation MSE (curvature²)')
baseline_curve = organoid_mse_summary(baseline_df).iloc[0]
axes[0].axhline(baseline_curve['mean'], color='black', linestyle='--', label='Global baseline')
if pd.notna(baseline_curve['sem']):
    axes[0].axhspan(baseline_curve['mean'] - baseline_curve['sem'],
                   baseline_curve['mean'] + baseline_curve['sem'], color='black', alpha=.12)
axes[0].set_title('Validation MSE: mean ± SEM across organoids')
axes[0].legend(fontsize=7)
axes[1].set(xlabel='Epoch', ylabel='Validation MAE (training target units)')
plt.tight_layout(); plt.show()
# Reopen independently, with no training or preprocessing fit:
saved = AnalysisRun(RUN_DIR)
display(saved.records)
restored = saved.select(saved.records.iloc[0].key)
print('Restored validation organoids:', len(restored['groups']['val']))


## Validation MSE by anatomical region and model depth

Reopen every saved checkpoint and evaluate its validation cells with the saved inputs, target transform, and baseline. Regions are derived directly from the source dataset's `d_crypts_graph` and `proj_vertex_ids` arrays and the original segmentation file named in each organoid's metadata. No tables from another experiment are loaded. Annotation settings are near the top of this notebook.

Use the **original nearest crypt**, checked against segmentation distances at each cell's projected vertex. Its circumference must have a supported local minimum or flat section near s=1 and enough assigned cells. Within these qualified territories, s<0.8 is crypt body, 0.8≤s≤1.2 is neck, and s>1.2 is the operational villus distance proxy (thresholds are configurable). Cells are never reassigned from an unqualified crypt to a farther qualified one. Missing annotations, no detected crypts, and unqualified territories remain separate in the support and score tables, rather than being labelled villus. Small undetected crypts remain a limitation of the source annotations.

Each panel shows physical-curvature MSE against depth for all saved folds, with separate curves for model variants and a dashed global baseline. First average squared errors within each organoid/region, then average repeated seeds or validation appearances within each organoid. Shading is ±1 SEM across organoids, conditional on the fitted models. A single trained depth produces a single point; SEM is absent with fewer than two organoids. Masking models are evaluated with all fates observed. Model inputs and fitting are unchanged.

Save per-organoid MSE, node-aligned region assignments and their source paths, support counts, summaries, settings, and the figure under the training run's `regional_evaluation/` folder. This section can also be rerun on an existing `RUN_DIR` without retraining, after restoring imports, `DEVICE`, and `REGION_SETTINGS`.


In [ ]:
from src.analysis.spatial.regions import graph_regions
from src.analysis.metrics.evaluation import prediction_table, regional_mse
from src.plotting.depth_scan import plot_regional_mse

regional_run = AnalysisRun(RUN_DIR)
region_dataset_name = (regional_run.settings['dataset'] if regional_run.modern
                       else regional_run.legacy.settings['DATASET_NAME'])
region_data_dir = ROOT / 'training_data' / region_dataset_name
region_output = RUN_DIR / 'regional_evaluation'
region_output.mkdir(exist_ok=True)
(region_output / 'settings.json').write_text(json.dumps(
    dict(dataset=str(region_data_dir), **REGION_SETTINGS), indent=2))
region_annotations = {}
region_score_parts = []
for regional_record in regional_run.records.to_dict('records'):
    selected = regional_run.select(regional_record['key'], device=DEVICE)
    for graph in selected['groups']['val']:
        if graph.organoid_str not in region_annotations:
            raw_graph = selected['raw_graphs'][graph.organoid_str]
            region_annotations[graph.organoid_str] = graph_regions(raw_graph, region_data_dir, **REGION_SETTINGS)
    annotations = pd.concat([region_annotations[g.organoid_str] for g in selected['groups']['val']], ignore_index=True)
    predictions = prediction_table(selected, device=DEVICE)
    if 'baseline_squared_error' not in predictions:
        raise ValueError('Saved per-node baseline predictions are required for regional comparison.')
    regional_scores = regional_mse(predictions, annotations)
    for column in ('key', 'name', 'subset', 'signal', 'hidden_dim', 'depth', 'fold', 'seed', 'rate'):
        if column in regional_record:
            regional_scores[column] = regional_record[column]
    region_score_parts.append(regional_scores)
    del selected, predictions
regional_scores = pd.concat(region_score_parts, ignore_index=True)
regional_scores.to_csv(region_output / 'validation_mse.csv', index=False)
annotations = pd.concat(region_annotations.values(), ignore_index=True)
annotations.to_csv(region_output / 'node_regions.csv.gz', index=False)
region_support = annotations.groupby(['region', 'annotation_status']).agg(
    organoids=('organoid_str', 'nunique'), cells=('node', 'size'))
region_support.to_csv(region_output / 'annotation_support.csv')
display(region_support)
variant_columns = [c for c in ('name', 'subset', 'signal', 'hidden_dim', 'rate', 'depth', 'region') if c in regional_scores]
regional_summary = organoid_mse_summary(regional_scores, variant_columns)
regional_summary.to_csv(region_output / 'mse_summary.csv', index=False)
regional_baseline_summary = organoid_mse_summary(regional_scores, ['depth', 'region'], value='baseline_mse')
regional_baseline_summary.to_csv(region_output / 'baseline_mse_summary.csv', index=False)
fig = plot_regional_mse(regional_scores)
fig.savefig(region_output / 'mse_by_depth.png', dpi=160, bbox_inches='tight')
plt.show()
